# Sesión 1: Indexación

En esta sesión construimos la base del sistema RAG basado en contenido para Steam. A partir de las reseñas generamos un documento textual por juego, calculamos embeddings semánticos y creamos un índice FAISS que servirá para la recuperación en la siguiente sesión.

## 2. Imports

In [16]:
import ast
import os
import re
import warnings

import faiss
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

## 3. Definición de rutas

In [17]:
USER_ITEMS_PATH = "../data/raw/steam_user.json"
REVIEWS_PATH = "../data/raw/steam_reviewers.json"
OUTPUTS_DIR = "../outputs"

for path in [USER_ITEMS_PATH, REVIEWS_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"No se encontró el archivo {path}. Ejecuta este notebook desde la carpeta notebooks/."
        )

os.makedirs(OUTPUTS_DIR, exist_ok=True)

## 4. Lectura y limpieza de texto

In [18]:
def read_json_lines(path, max_lines=None):
    """Lee un archivo pseudo-JSONL línea a línea usando ast.literal_eval."""
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_lines is not None and i >= max_lines:
                break

            line = line.strip()
            if not line:
                continue

            try:
                records.append(ast.literal_eval(line))
            except (SyntaxError, ValueError):
                continue

    return records


def clean_text(text):
    """Limpia texto eliminando HTML, saltos de línea y caracteres raros."""
    if text is None:
        return ""

    text = BeautifulSoup(str(text), "html.parser").get_text(" ")
    text = text.lower()
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"[^\w\s]+", " ", text, flags=re.UNICODE)
    text = re.sub(r"_+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_item_name_mapping(user_items_path):
    """Construye un diccionario item_id -> item_name usando steam_user.json."""
    item_name_map = {}
    records = read_json_lines(user_items_path)

    for user_record in tqdm(records, desc="Leyendo items de usuarios"):
        items = user_record.get("items", [])
        if not isinstance(items, list):
            continue

        for item_data in items:
            if not isinstance(item_data, dict):
                continue

            item_id = str(item_data.get("item_id", "")).strip()
            item_name = str(item_data.get("item_name", "")).strip()

            if not item_id or not item_name or item_id in item_name_map:
                continue

            item_name_map[item_id] = item_name

    return item_name_map

## 5. Construcción de documentos por juego

In [19]:
def build_game_documents(reviews_path, item_name_map, min_reviews=2, max_reviews_per_game=20, max_games=5000):
    """Construye un documento por juego a partir del nombre real y sus reseñas."""
    records = read_json_lines(reviews_path)
    grouped_reviews = {}

    for user_record in tqdm(records, desc="Procesando reseñas"):
        reviews = user_record.get("reviews", [])
        if not isinstance(reviews, list):
            continue

        for review_data in reviews:
            if not isinstance(review_data, dict):
                continue

            item_id = str(review_data.get("item_id", "")).strip()
            if not item_id:
                continue

            item_name = item_name_map.get(item_id)
            if not item_name:
                continue

            review_text = clean_text(review_data.get("review", ""))
            recommend = review_data.get("recommend", None)

            if not review_text:
                continue

            key = (item_id, item_name)
            if key not in grouped_reviews:
                grouped_reviews[key] = {
                    "positive_reviews": [],
                    "other_reviews": [],
                }

            if recommend is True:
                grouped_reviews[key]["positive_reviews"].append(review_text)
            else:
                grouped_reviews[key]["other_reviews"].append(review_text)

    rows = []
    for (item_id, item_name), reviews_data in tqdm(grouped_reviews.items(), desc="Construyendo documentos"):
        positive_reviews = reviews_data["positive_reviews"]
        other_reviews = reviews_data["other_reviews"]

        selected_reviews = positive_reviews[:max_reviews_per_game]
        if len(selected_reviews) < min_reviews:
            remaining_slots = max_reviews_per_game - len(selected_reviews)
            selected_reviews += other_reviews[:remaining_slots]

        if len(selected_reviews) < min_reviews:
            continue

        document_parts = [clean_text(item_name)] + selected_reviews
        document = " ".join(part for part in document_parts if part).strip()

        rows.append(
            {
                "item_id": item_id,
                "item_name": item_name,
                "num_reviews": len(selected_reviews),
                "document": document,
            }
        )

    columns = ["item_id", "item_name", "num_reviews", "document"]
    games_df = pd.DataFrame(rows, columns=columns)
    if games_df.empty:
        return games_df

    games_df = games_df.sort_values(
        by=["num_reviews", "item_name"],
        ascending=[False, True],
        kind="stable",
    ).head(max_games).reset_index(drop=True)

    return games_df

## 6. Crear el DataFrame de juegos

In [20]:
item_name_map = build_item_name_mapping(USER_ITEMS_PATH)

games_df = build_game_documents(
    REVIEWS_PATH,
    item_name_map,
    min_reviews=2,
    max_reviews_per_game=20,
    max_games=5000,
)

if games_df.empty:
    raise ValueError(
        "No se han generado documentos de juegos. Revisa los datos o reduce min_reviews."
    )

Construyendo documentos: 100%|██████████| 3319/3319 [00:00<00:00, 38144.56it/s]


## 7. Inspección rápida del dataset

In [21]:
games_df.head()

,item_id,item_name,num_reviews,document
0,251570,7 Days to Die,20,7 days to die this game is awesome its like mi...
1,250420,8BitMMO,20,8bitmmo this game is so much fun i love buildi...
2,113400,APB Reloaded,20,apb reloaded i do hate the fact that i lagged ...
3,346110,ARK: Survival Evolved,20,ark survival evolved so i play ark on and off ...
4,224540,Ace of Spades,20,ace of spades meh awesome great game yeah a pr...


In [22]:
games_df["item_name"].head(20)

0                                7 Days to Die
1                                      8BitMMO
2                                 APB Reloaded
3                        ARK: Survival Evolved
4                                Ace of Spades
5                         AdVenture Capitalist
6                Age of Empires II: HD Edition
7     Age of Empires® III: Complete Collection
8           Age of Mythology: Extended Edition
9                                      AirMech
10                                   Alan Wake
11                            Alien: Isolation
12                         Aliens vs. Predator
13                   Amnesia: The Dark Descent
14                                 Antichamber
15                                  Archeblade
16                            Arma 2: DayZ Mod
17                 Arma 2: Operation Arrowhead
18                                      Arma 3
19                         Assassin's Creed II
Name: item_name, dtype: object

In [23]:
fake_names = games_df["item_name"].astype(str).str.startswith("item_").sum()
print("Nombres placeholder:", fake_names)

Nombres placeholder: 0


In [24]:
games_df.shape

(2058, 4)

In [25]:
games_df["num_reviews"].describe()

count    2058.000000
mean        7.817298
std         6.866294
min         2.000000
25%         2.000000
50%         4.000000
75%        13.000000
max        20.000000
Name: num_reviews, dtype: float64

## 8. Guardar dataset procesado

In [26]:
games_dataset_path = os.path.join(OUTPUTS_DIR, "games_dataset.csv")
games_df.to_csv(games_dataset_path, index=False, encoding="utf-8")
print(f"Dataset guardado en: {games_dataset_path}")

Dataset guardado en: ../outputs\games_dataset.csv


## 9. Cargar modelo de embeddings

In [27]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6056.37it/s]


## 10. Generar embeddings

In [28]:
documents = games_df["document"].tolist()

embeddings = model.encode(
    documents,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

embeddings = np.asarray(embeddings, dtype=np.float32)
embeddings.shape

Batches: 100%|██████████| 33/33 [00:39<00:00,  1.20s/it]


(2058, 384)

## 11. Normalizar embeddings

In [29]:
faiss.normalize_L2(embeddings)
embeddings[:2, :5]

array([[-0.05179485,  0.01100873,  0.00551642, -0.05879765,  0.0040854 ],
       [-0.07396194, -0.01828658,  0.0492026 , -0.01372171, -0.04666809]],
      dtype=float32)

## 12. Crear índice FAISS

In [30]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
index.ntotal

2058

## 13. Guardar embeddings e índice

In [31]:
embeddings_path = os.path.join(OUTPUTS_DIR, "embeddings.npy")
index_path = os.path.join(OUTPUTS_DIR, "faiss.index")

np.save(embeddings_path, embeddings)
faiss.write_index(index, index_path)

print(f"Embeddings guardados en: {embeddings_path}")
print(f"Índice FAISS guardado en: {index_path}")

Embeddings guardados en: ../outputs\embeddings.npy
Índice FAISS guardado en: ../outputs\faiss.index


## 14. Prueba rápida de búsqueda

In [32]:
query = "story rich action adventure game with positive reviews"

query_embedding = model.encode([query], convert_to_numpy=True)
query_embedding = np.asarray(query_embedding, dtype=np.float32)
faiss.normalize_L2(query_embedding)

top_k = min(5, len(games_df))
scores, indices = index.search(query_embedding, top_k)

results_df = games_df.iloc[indices[0]][["item_id", "item_name"]].copy()
results_df["score"] = scores[0]
results_df = results_df.reset_index(drop=True)
results_df

,item_id,item_name,score
0,278360,A Story About My Uncle,0.651342
1,362680,Fran Bow,0.628994
2,232430,Gone Home,0.619686
3,263680,Unearthed: Trail of Ibn Battuta - Episode 1 - ...,0.616961
4,203810,Dear Esther,0.593848


In [33]:
print("Resumen sesión 1")
print(f"Número de juegos indexados: {len(games_df)}")
print(f"Dimensión de embeddings: {embeddings.shape[1]}")
print(f"Número de vectores en FAISS: {index.ntotal}")
print(f"Nombres placeholder: {games_df['item_name'].str.startswith('item_').sum()}")

Resumen sesión 1
Número de juegos indexados: 2058
Dimensión de embeddings: 384
Número de vectores en FAISS: 2058
Nombres placeholder: 0


## 15. Conclusión

En esta sesión hemos generado el dataset de juegos con nombres reales, sus documentos textuales, los embeddings normalizados y el índice FAISS. Los archivos creados en `../outputs` son `games_dataset.csv`, `embeddings.npy` y `faiss.index`, que servirán como base para la recuperación en la sesión 2.

En la primera sesión se procesó el dataset de Steam para construir una colección de ítems recomendables. Dado que las reseñas no incluían el nombre del juego, se generó previamente un mapeo item_id → item_name a partir del archivo de usuarios e ítems. Posteriormente, se agregaron reseñas positivas por juego y se construyó un documento textual por ítem combinando el nombre real del juego con sus reseñas limpias. Estos documentos fueron transformados en embeddings semánticos mediante all-MiniLM-L6-v2, normalizados con norma L2 e indexados en FAISS usando IndexFlatIP, de forma que el producto interno equivale a similitud coseno.